AIMA-LAB - lezione 2: Modellazione di problemi

- 8-Game Puzzle
- N-Queen Problem
- Vacuum Cleaner

In [ ]:
# ATTENZIONE: eseguire questo blocco solo se si sta usando Google Colab. Se si lavora in locale basta estrarre in questa cartella la libreria aima.zip caricata su Unistudium
import sys, os

py_file_location = "/content/aima"
sys.path.append(os.path.abspath(py_file_location))

print("Caricare la libreria aima.zip come distribuita nella pagina Unistudium del corso")
from google.colab import files
upload = files.upload()

In [ ]:
# Estrai la libreria aima. Usare solo da Google Colab o se il comando unzip è installato
!unzip aima.zip

In [ ]:
# Installa le dipendenze nell'ambiente locale
%pip install numpy

In [ ]:
from aima import Problem, Node, memoize, PriorityQueue
from aima import depth_first_graph_search, breadth_first_graph_search, iterative_deepening_search, depth_limited_search
from collections.abc import Callable
import time

# Colori da usare nella print()
BLUE = "\033[34;1m"
RED = "\033[31;1m"
GREEN = "\033[32;1m"
RESET = "\033[0m"

In [ ]:
from dataclasses import dataclass

@dataclass
class Result:
    result: Node
    nodes_generated: int
    paths_explored: int
    nodes_left_in_frontier: int

def best_first_graph_search(problem: Problem, f: Callable) -> Result:
    f = memoize(f, 'f')
    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()
    counter = 1
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return Result(node, counter, len(explored), len(frontier))
        explored.add(node.state)
        for child in node.expand(problem):
            counter += 1
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                if f(child) < frontier[child]:
                    del frontier[child]
                    frontier.append(child)
    return Result(None, counter, len(explored), 0)

# Varianti della BestFirst utilizzando diverse funzioni di costo
def ucs(problem: Problem) -> Result:
  return best_first_graph_search(problem, lambda node : node.path_cost)

def greedy(problem: Problem, h: Callable | None = None) -> Result:
  h = memoize(h or problem.h, 'h')
  return best_first_graph_search(problem, h)

def astar(problem: Problem, h: Callable | None = None) -> Result:
  h = memoize(h or problem.h, 'h')
  return best_first_graph_search(problem, lambda node : h(node) + node.path_cost)

In [ ]:
# Funzione per eseguire gli algoritmi e stampare alcune informazioni come
# risultato, costi della soluzione e tempo di esecuzione.
def execute(name: str, algorithm: Callable, problem: Problem, *args, **kwargs) -> None:
    print(f"{RED}{name}{RESET}\n")
    start = time.time()
    sol = algorithm(problem, *args, **kwargs)
    end = time.time()
    if problem.goal is not None:
        print(f"\n{GREEN}PROBLEM:{RESET} {problem.initial} -> {problem.goal}")
    if isinstance(sol, Result):
        print(f"{GREEN}Total nodes generated:{RESET} {sol.nodes_generated}")
        print(f"{GREEN}Paths explored:{RESET} {sol.paths_explored}")
        print(f"{GREEN}Nodes left in frontier:{RESET} {sol.nodes_left_in_frontier}")
        sol = sol.result
    print(f"{GREEN}Result:{RESET} {sol.solution() if sol is not None else '---'}")
    if isinstance(sol, Node):
        print(f"{GREEN}Path Cost:{RESET} {sol.path_cost}")
        print(f"{GREEN}Path Length:{RESET} {sol.depth}")
    print(f"{GREEN}Time:{RESET} {end - start} s")

In [ ]:
class EightPuzzle(Problem):
    """ The problem of sliding tiles numbered from 1 to 8 on a 3x3 board, where one of the
    squares is a blank. A state is represented as a tuple of length 9, where  element at
    index i represents the tile number  at index i (0 if it's an empty square) """

    def __init__(self, initial: tuple, goal: tuple = (1, 2, 3, 4, 5, 6, 7, 8, 0)) -> None:
        """ Define goal state and initialize a problem """


    def actions(self, state: tuple) -> list[str]:
        """ Return the actions that can be executed in the given state.
        The result would be a list, since there are only four possible actions
        in any given state of the environment """

        return possible_actions

    def result(self, state: tuple, action: str) -> tuple:
        """ Given state and action, return a new state that is the result of the action.
        Action is assumed to be a valid action in the state """



    def goal_test(self, state: tuple) -> bool:
        """ Given a state, return True if state is a goal state or False, otherwise """


    # Implementare le seguenti euristiche dopo aver fatto A*

    def misplaced_tiles(self, node: Node) -> int:
        """ Return the heuristic value for a given state. Default heuristic function used is
        h(n) = number of misplaced tiles """


    def manh(self, node: Node) -> int:
        """The Manhattan heuristic."""


In [ ]:
p1 = EightPuzzle((1,2,3,0,4,6,7,5,8))
p2 = EightPuzzle((7,5,4,0,3,2,8,1,6))
p3 = EightPuzzle((7,2,4,5,0,6,8,3,1))

In [ ]:
import math

class VacuumState:

    # def __init__

    def get_N(self) -> int:
        n = math.floor(math.sqrt(len(self.environment)))
        if n ** 2 != len(self.environment): raise ValueError("Invalid environment size. It must be a square")
        return n

        def __str__(self) -> str:
      return str(f"VacuumState(pos={self.cleaner_pos}, environment={self.environment})")

In [ ]:
class VacuumCleaner(Problem):
    """The problem of navigating a vacuum cleaner on an NxN grid where each cell can be
    clean, dirty, or contain a wall. A state is represented as an N-element array,
    where each entry can be 'C' for clean, 'D' for dirty, or 'W' for wall and the cleaner position.
    The vacuum cleaner can move to any adjacent cell (as long as it does not move outside the grid
    or into a wall) or clean the current cell.
    """

    def __init__(self, initial: VacuumState) -> None:
        """ Define goal state and initialize a problem """

    def actions(self, state: VacuumState) -> list[str]:
        """ Return the actions that can be executed in the given state.
        The result would be a list, since there are only five possible actions
        in any given state of the environment """


        return possible_actions

    def has_wall(self, state: VacuumState, action: str) -> bool:
        """ Test if making an action in a certain state bring to a wall """


    def result(self, state: VacuumState, action: str) -> tuple:
        """ Given state and action, return a new state that is the result of the action.
        Action is assumed to be a valid action in the state """



    def goal_test(self, state: VacuumState) -> bool:
        """ Given a state, return True if state is a goal state or False, otherwise """



In [ ]:
state1 = VacuumState(0, ('C', 'D', 'W', 'W', 'C', 'D', 'D', 'C', 'W'))
state2 = VacuumState(5, ('C', 'D', 'C', 'D', 'W', 'C', 'W', 'D', 'D', 'D', 'W', 'C', 'W', 'C', 'C', 'D'))

cleaner = VacuumCleaner(state1)

print(cleaner.initial)
for i, t in enumerate(cleaner.initial.environment, start=1):
    print(t, end=' ')
    if i % cleaner.initial.get_N() == 0: print()

In [ ]:
class NQueensProblem(Problem):
    """The problem of placing N queens on an NxN board with none attacking
    each other. A state is represented as an N-element array, where
    a value of r in the c-th entry means there is a queen at column c,
    row r, and a value of -1 means that the c-th column has not been
    filled in yet. We fill in columns left to right.
    """

    def __init__(self, N: int) -> None:
        super().__init__(tuple([-1] * N))
        self.N = N

    def actions(self, state: tuple) -> list[int]:
        """In the leftmost empty column, try all non-conflicting rows."""
        if state[-1] != -1:
            return []  # All columns filled; no successors
        else:
            col = state.index(-1)
            return [row for row in range(self.N)
                    if not self.conflicted(state, row, col)]

    def result(self, state: tuple, row: int) -> tuple:
        """Place the next queen at the given row."""
        col = state.index(-1)
        new = list(state[:])
        new[col] = row
        return tuple(new)

    def conflicted(self, state: tuple, row: int, col: int) -> bool:
        """Would placing a queen at (row, col) conflict with anything?"""
        return any(self.conflict(row, col, state[c], c)
                   for c in range(col))

    def conflict(self, row1: int, col1: int, row2: int, col2: int) -> bool:
        """Would putting two queens in (row1, col1) and (row2, col2) conflict?"""
        return (row1 == row2 or  # same row
                col1 == col2 or  # same column
                row1 - col1 == row2 - col2 or  # same \ diagonal
                row1 + col1 == row2 + col2)  # same / diagonal

    def goal_test(self, state: tuple) -> bool:
        """Check if all columns filled, no conflicts."""
        if state[-1] == -1:
            return False
        return not any(self.conflicted(state, state[col], col)
                       for col in range(len(state)))